## 3.05 Boolean Expressions

SFI Safety Tech Inspector: every check in this notebook is a question about an SFI automotive safety record, and every answer is only `True` or `False`. The plan is to build small named Boolean checks first, then combine them with `and`, `or`, and `not` into one backend decision.

Quick reminders from the lesson:

- `=` assigns a value, `==` compares two values.
- "All requirements must pass" means `and`. "Either option is fine" means `or`. "Reject a duplicate" means `not duplicate`.
- Test valid, invalid, and edge cases, not just one record.


## Popcorn Hack 1: Check One SFI Part Record

A record is complete when it has a product name, a racing category, a spec number, and an effective date. Only **Auto Racing** and **Drag Racing** count as valid categories.

Each check is its own named Boolean. "Non-empty" is written as `!= ""`, and "valid category" uses `in`, which is True when the category matches one of the list items.


In [ ]:
# CODE_RUNNER: 3.05 Popcorn 1 - check SFI part fields

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]

has_product_name = part["product_name"] != ""
valid_category = part["category"] in valid_categories
has_spec_number = part["spec_number"] != ""
has_effective_date = part["effective_date"] != ""
record_complete = has_product_name and valid_category and has_spec_number and has_effective_date

print("Has product name:", has_product_name)
print("Valid category:", valid_category)
print("Has spec number:", has_spec_number)
print("Has effective date:", has_effective_date)
print("Record complete:", record_complete)


In [ ]:
# CODE_RUNNER: 3.05 Popcorn 1 - changed category to Street Car

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Street Car",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]

has_product_name = part["product_name"] != ""
valid_category = part["category"] in valid_categories
has_spec_number = part["spec_number"] != ""
has_effective_date = part["effective_date"] != ""
record_complete = has_product_name and valid_category and has_spec_number and has_effective_date

print("Has product name:", has_product_name)
print("Valid category:", valid_category)
print("Has spec number:", has_spec_number)
print("Has effective date:", has_effective_date)
print("Record complete:", record_complete)


## Popcorn Hack 2: Backend Create Rule

A new database record should only be created when all required fields are valid **and** its spec number is not already stored.

    can_create = fields_valid and not duplicate_spec

In [ ]:
# CODE_RUNNER: 3.05 Popcorn 2 - rejected case (duplicate spec 1.1)

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]

fields_valid = (
    part["product_name"] != ""
    and part["category"] in valid_categories
    and part["spec_number"] != ""
    and part["effective_date"] != ""
)
duplicate_spec = part["spec_number"] in existing_spec_numbers
can_create = fields_valid and not duplicate_spec

print("Spec", part["spec_number"], "| fields valid:", fields_valid, "| duplicate:", duplicate_spec)
if can_create:
    print("CREATE record")
else:
    print("REJECT record")


In [ ]:
# CODE_RUNNER: 3.05 Popcorn 2 - valid case (new spec 1.3)

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.3",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]

fields_valid = (
    part["product_name"] != ""
    and part["category"] in valid_categories
    and part["spec_number"] != ""
    and part["effective_date"] != ""
)
duplicate_spec = part["spec_number"] in existing_spec_numbers
can_create = fields_valid and not duplicate_spec

print("Spec", part["spec_number"], "| fields valid:", fields_valid, "| duplicate:", duplicate_spec)
if can_create:
    print("CREATE record")
else:
    print("REJECT record")


## Popcorn Hack 3: SFI Search Filter

A record matches when the query appears in its product name **or** equals its spec number exactly. It is only returned if it also has an accepted category.

    name_match = query.lower() in product_name.lower()
    spec_match = query == spec_number
    include = (name_match or spec_match) and category in valid_categories

The parentheses matter: the `or` is decided first, then the category check must also pass.

In [ ]:
# CODE_RUNNER: 3.05 Popcorn 3 - search by spec number

query = "1.2"

records = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

valid_categories = ["Auto Racing", "Drag Racing"]

match_count = 0
for record in records:
    name_match = query.lower() in record["product_name"].lower()
    spec_match = query == record["spec_number"]
    accepted_category = record["category"] in valid_categories
    if (name_match or spec_match) and accepted_category:
        print("MATCH:", record["spec_number"], "-", record["product_name"])
        match_count = match_count + 1

print("Results:", match_count)


In [ ]:
# CODE_RUNNER: 3.05 Popcorn 3 - search by name, Street Car filtered out

query = "flywheel"

records = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    },
    {
        "product_name": "Street Flywheel Kit",
        "category": "Street Car",
        "spec_number": "9.9"
    }
]

valid_categories = ["Auto Racing", "Drag Racing"]

match_count = 0
for record in records:
    name_match = query.lower() in record["product_name"].lower()
    spec_match = query == record["spec_number"]
    accepted_category = record["category"] in valid_categories
    if (name_match or spec_match) and accepted_category:
        print("MATCH:", record["spec_number"], "-", record["product_name"])
        match_count = match_count + 1

print("Results:", match_count)


## In-class MCQ Check

| # | Question | Answer | Why |
|---|---|---|---|
| 1 | What does `7 >= 7` evaluate to? | **A. True** | `>=` includes equality, and 7 equals 7. |
| 2 | A record is valid only when it has a name AND a spec number. Which operator? | **C. and** | Both requirements must be true. |
| 3 | `duplicate_spec` is `False`. What is `not duplicate_spec`? | **A. True** | `not` reverses the Boolean value. |
| 4 | Which expression accepts either Auto Racing or Drag Racing? | **B.** `category == 'Auto Racing' or category == 'Drag Racing'` | Only one of the two comparisons has to be true. Choice A with `and` can never be True, since a category cannot equal both strings at once. |

**Result line:**

MCQ 3.05: 4/4 | answers: A,C,A,B


## Homework Hack: SFI Backend Validator

A valid record has a non-empty product name, an accepted racing category, a non-empty spec number, an effective date, and a spec number that does **not** already exist in the backend.

    is_valid = has_product_name and valid_category and has_spec_number and has_effective_date and not duplicate_spec

Every rule is required, so they are joined with `and`. The duplicate rule is a rejection rule, so it is flipped with `not`: a record passes that part only when `duplicate_spec` is `False`. A second function, `rejection_reasons`, reuses the same checks so the output says *why* a record was rejected, not just that it was.

The validator is a function, so it is reusable: the same code checks the three supplied records and then the extra edge cases below. The cell is self-contained and runs without any earlier cell.

1. Spec `1.2`, Auto Racing, all fields present, not stored yet → **ACCEPT**.
2. Spec `3.1`, but category is **Street Car** → **REJECT** (category).
3. Spec `1.1`, Drag Racing, all fields present, but `1.1` is already stored → **REJECT** (duplicate).


In [ ]:
# CODE_RUNNER: 3.05 Homework - SFI Backend Validator

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]


def check_rules(record):
    has_product_name = record["product_name"] != ""
    valid_category = record["category"] in valid_categories
    has_spec_number = record["spec_number"] != ""
    has_effective_date = record["effective_date"] != ""
    duplicate_spec = record["spec_number"] in existing_spec_numbers
    return has_product_name, valid_category, has_spec_number, has_effective_date, duplicate_spec


def is_valid_record(record):
    has_product_name, valid_category, has_spec_number, has_effective_date, duplicate_spec = check_rules(record)
    return (
        has_product_name
        and valid_category
        and has_spec_number
        and has_effective_date
        and not duplicate_spec
    )


def rejection_reasons(record):
    has_product_name, valid_category, has_spec_number, has_effective_date, duplicate_spec = check_rules(record)
    reasons = []
    if not has_product_name:
        reasons.append("missing product name")
    if not valid_category:
        reasons.append("category not accepted")
    if not has_spec_number:
        reasons.append("missing spec number")
    if not has_effective_date:
        reasons.append("missing effective date")
    if duplicate_spec:
        reasons.append("duplicate spec number")
    return reasons


def validate_all(records):
    accepted = 0
    for record in records:
        label = "Spec " + record["spec_number"] if record["spec_number"] != "" else "Spec (blank)"
        if is_valid_record(record):
            print("ACCEPT |", label, "|", record["product_name"])
            accepted = accepted + 1
        else:
            print("REJECT |", label, "|", ", ".join(rejection_reasons(record)))
    print("Accepted", accepted, "of", len(records))


test_records = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.2",
        "effective_date": "Feb. 9, 2006"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Street Car",
        "spec_number": "3.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Existing Flywheel Record",
        "category": "Drag Racing",
        "spec_number": "1.1",
        "effective_date": "Jan. 1, 2026"
    }
]

validate_all(test_records)


## Tests: edge cases

Test missing fields and records that violate several rules at once.

| Test record | Expected result |
|---|---|
| Empty product name, Drag Racing, spec `4.1`, has a date | REJECT, missing product name |
| Auto Racing, spec `5.1`, **empty** effective date | REJECT, missing effective date |
| Empty spec number, Auto Racing, has a date | REJECT, missing spec number |
| Spec `2.1` (stored) **and** category `"Karting"` | REJECT, category not accepted **and** duplicate spec number |
| Spec `2.2`, Drag Racing, all fields present | ACCEPT (boundary check: `2.2` is close to `2.1` but is not stored, and `==`/`in` match exact strings) |


In [ ]:
# CODE_RUNNER: 3.05 Tests - SFI validator edge cases

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]


def check_rules(record):
    has_product_name = record["product_name"] != ""
    valid_category = record["category"] in valid_categories
    has_spec_number = record["spec_number"] != ""
    has_effective_date = record["effective_date"] != ""
    duplicate_spec = record["spec_number"] in existing_spec_numbers
    return has_product_name, valid_category, has_spec_number, has_effective_date, duplicate_spec


def is_valid_record(record):
    has_product_name, valid_category, has_spec_number, has_effective_date, duplicate_spec = check_rules(record)
    return (
        has_product_name
        and valid_category
        and has_spec_number
        and has_effective_date
        and not duplicate_spec
    )


def rejection_reasons(record):
    has_product_name, valid_category, has_spec_number, has_effective_date, duplicate_spec = check_rules(record)
    reasons = []
    if not has_product_name:
        reasons.append("missing product name")
    if not valid_category:
        reasons.append("category not accepted")
    if not has_spec_number:
        reasons.append("missing spec number")
    if not has_effective_date:
        reasons.append("missing effective date")
    if duplicate_spec:
        reasons.append("duplicate spec number")
    return reasons


def validate_all(records):
    accepted = 0
    for record in records:
        label = "Spec " + record["spec_number"] if record["spec_number"] != "" else "Spec (blank)"
        if is_valid_record(record):
            print("ACCEPT |", label, "|", record["product_name"])
            accepted = accepted + 1
        else:
            print("REJECT |", label, "|", ", ".join(rejection_reasons(record)))
    print("Accepted", accepted, "of", len(records))


edge_records = [
    {"product_name": "", "category": "Drag Racing", "spec_number": "4.1", "effective_date": "Mar. 3, 2026"},
    {"product_name": "Clutch Cover", "category": "Auto Racing", "spec_number": "5.1", "effective_date": ""},
    {"product_name": "Flywheel Shield", "category": "Auto Racing", "spec_number": "", "effective_date": "Apr. 1, 2026"},
    {"product_name": "Kart Flywheel", "category": "Karting", "spec_number": "2.1", "effective_date": "May 5, 2026"},
    {"product_name": "Bellhousing", "category": "Drag Racing", "spec_number": "2.2", "effective_date": "Jun. 6, 2026"}
]

validate_all(edge_records)


## Pseudocode Check

The lesson's College Board version of the validator, with my edge-case record **Kart Flywheel** (category `"Karting"`, spec `"2.1"`) traced through it:

**Python result:** the edge-case cell printed `REJECT | Spec 2.1 | category not accepted, duplicate spec number`, so both versions make the same decision.

**Differences to remember:** pseudocode compares with `=` and uses `≠`, while Python uses `==` and `!=`. Pseudocode writes `AND`, `OR`, `NOT` in capitals; Python uses `and`, `or`, `not`. The pseudocode checks categories with two `=` comparisons joined by `OR`; my Python uses `in valid_categories`, which asks the same "is it one of these?" question.


## Submission Notes